# Week 5 — Fourier Series, DFT, FFT & Laplace Transforms

> **Differential Equations for Scientists & Engineers**  
> *From Fourier's heat trick to the Cooley-Tukey butterfly — every algorithm derived from scratch.*

---

## Learning Objectives

1. Derive **Fourier series** coefficients from orthogonality of sines and cosines
2. Analyse **Gibbs phenomenon** and pointwise convergence
3. Implement the **Discrete Fourier Transform (DFT)** in $O(N^2)$ from first principles
4. Derive and implement the **Cooley-Tukey FFT** in $O(N\log N)$
5. Apply the **Laplace transform** to solve ODEs and analyse transfer functions
6. Use the **convolution theorem** to multiply polynomials with FFT


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. Fourier Series — Derivation

For a function $f$ periodic on $[-L, L]$, expand in terms of the orthonormal basis $\{1, \cos(n\pi x/L), \sin(n\pi x/L)\}_{n\geq 1}$.

Using orthogonality $\int_{-L}^{L} \cos(m\pi x/L)\cos(n\pi x/L)\,dx = L\delta_{mn}$:

$$\boxed{f(x) = \frac{a_0}{2} + \sum_{n=1}^{\infty}\left[a_n\cos\frac{n\pi x}{L} + b_n\sin\frac{n\pi x}{L}\right]}$$

$$a_n = \frac{1}{L}\int_{-L}^{L}f(x)\cos\frac{n\pi x}{L}\,dx, \quad b_n = \frac{1}{L}\int_{-L}^{L}f(x)\sin\frac{n\pi x}{L}\,dx$$

In [ ]:
def fourier_coefficients(f, L, N, n_quad=2000):
    """Compute Fourier coefficients a_0,...,a_N, b_1,...,b_N via numerical quadrature."""
    x = np.linspace(-L, L, n_quad, endpoint=False)
    fx = f(x)
    dx = 2*L / n_quad

    a = np.zeros(N+1)
    b = np.zeros(N+1)
    a[0] = np.sum(fx) * dx / L
    for n in range(1, N+1):
        a[n] = np.sum(fx * np.cos(n*np.pi*x/L)) * dx / L
        b[n] = np.sum(fx * np.sin(n*np.pi*x/L)) * dx / L
    return a, b


def fourier_reconstruction(a, b, L, x, N=None):
    """Reconstruct function from Fourier coefficients."""
    if N is None: N = len(a) - 1
    result = a[0] / 2
    for n in range(1, N+1):
        result = result + a[n]*np.cos(n*np.pi*x/L) + b[n]*np.sin(n*np.pi*x/L)
    return result


# --- Square wave and Gibbs phenomenon ---
L = np.pi
f_square = lambda x: np.sign(np.sin(x + 1e-10))

x_plot = np.linspace(-L, L, 1000)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x_plot, f_square(x_plot), 'k-', lw=1.5, label='Square wave')
colors = cm.plasma(np.linspace(0.15, 0.85, 5))
for N_terms, c in zip([1, 3, 7, 15, 51], colors):
    a, b = fourier_coefficients(f_square, L, N_terms)
    y_rec = fourier_reconstruction(a, b, L, x_plot)
    axes[0].plot(x_plot, y_rec, color=c, lw=1, alpha=0.8, label=f'N={N_terms}')
axes[0].set_title('Fourier Series Convergence (Square Wave)')
axes[0].legend(frameon=False, fontsize=8)

# Gibbs overshoot measurement
N_gibbs = np.arange(1, 101, 2)
overshoot = []
for N_t in N_gibbs:
    a, b = fourier_coefficients(f_square, L, N_t)
    y_rec = fourier_reconstruction(a, b, L, x_plot)
    overshoot.append(np.max(y_rec) - 1.0)

axes[1].plot(N_gibbs, overshoot, 'r-', lw=2)
axes[1].axhline(0.0895, color='k', ls='--', label='Gibbs limit ≈ 8.95%')
axes[1].set_xlabel('Number of terms N')
axes[1].set_ylabel('Overshoot')
axes[1].set_title('Gibbs Phenomenon — Overshoot vs N')
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 2. Discrete Fourier Transform — From First Principles

Given $N$ complex samples $x_0, \ldots, x_{N-1}$, the **DFT** is:

$$X_k = \sum_{n=0}^{N-1} x_n\, e^{-2\pi i kn/N}, \quad k = 0, 1, \ldots, N-1$$

This is an $N\times N$ matrix-vector product: $\mathbf{X} = W\mathbf{x}$ where $W_{kn} = \omega^{kn}$, $\omega = e^{-2\pi i/N}$. Cost: $O(N^2)$.

In [ ]:
def dft_naive(x):
    """O(N^2) DFT via explicit matrix multiplication."""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return W @ x


def fft_cooley_tukey(x):
    """
    Radix-2 Cooley-Tukey FFT — recursive O(N log N) implementation.
    N must be a power of 2.
    """
    N = len(x)
    if N <= 1:
        return x
    if N % 2 != 0:
        raise ValueError('N must be a power of 2')

    # Divide
    even = fft_cooley_tukey(x[::2])
    odd  = fft_cooley_tukey(x[1::2])

    # Twiddle factors
    twiddle = np.exp(-2j * np.pi * np.arange(N//2) / N)

    # Butterfly combination
    return np.concatenate([
        even + twiddle * odd,
        even - twiddle * odd
    ])


# Correctness check
x_test = np.random.randn(64) + 1j * np.random.randn(64)
X_naive   = dft_naive(x_test)
X_fft     = fft_cooley_tukey(x_test)
X_numpy   = np.fft.fft(x_test)
print(f"DFT vs FFT max error:   {np.max(np.abs(X_naive - X_fft)):.2e}")
print(f"FFT vs NumPy max error: {np.max(np.abs(X_fft - X_numpy)):.2e}")

# Timing comparison
N_time = 1024
x_big = np.random.randn(N_time)

t0 = time.time(); dft_naive(x_big); t_naive = time.time() - t0
t0 = time.time(); fft_cooley_tukey(x_big); t_fft = time.time() - t0
t0 = time.time(); np.fft.fft(x_big); t_np = time.time() - t0

print(f"\nN={N_time}:")
print(f"  Naive DFT O(N²):      {t_naive*1000:.1f} ms")
print(f"  Cooley-Tukey FFT:     {t_fft*1000:.1f} ms")
print(f"  NumPy FFT (C/FFTW):   {t_np*1000:.2f} ms")

---

## 3. Spectral Analysis Application — Signal Decomposition

In [ ]:
# Compose a signal: two sinusoids + noise
fs = 1000        # sampling frequency Hz
T_sig = 1.0      # duration seconds
t = np.linspace(0, T_sig, int(fs * T_sig), endpoint=False)

f1, f2 = 50, 120   # Hz
signal = (1.5*np.sin(2*np.pi*f1*t) + 0.7*np.sin(2*np.pi*f2*t) +
          0.5*np.random.randn(len(t)))

# FFT
X = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(t), d=1/fs)
magnitude = np.abs(X) / len(t)

# Keep only positive frequencies
pos = freqs >= 0
freqs_pos = freqs[pos]; mag_pos = 2 * magnitude[pos]

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(t[:200], signal[:200], color='#1565C0', lw=1)
axes[0].set_title('Time-domain signal (first 200 samples)')
axes[0].set_xlabel('Time (s)')

axes[1].plot(freqs_pos, mag_pos, color='#B71C1C', lw=1.5)
axes[1].axvline(f1, color='green', ls='--', lw=1.5, label=f'{f1} Hz')
axes[1].axvline(f2, color='orange', ls='--', lw=1.5, label=f'{f2} Hz')
axes[1].set_xlim(0, 300)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Amplitude')
axes[1].set_title('Magnitude Spectrum (FFT)')
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 4. Laplace Transforms for ODEs

The **Laplace transform** $\mathcal{L}\{f\}(s) = \int_0^\infty f(t)e^{-st}\,dt$ converts ODE problems to algebraic ones.

Key properties used to solve ODEs:

| Property | Formula |
|---|---|
| Derivative | $\mathcal{L}\{f'\} = sF(s) - f(0)$ |
| 2nd Derivative | $\mathcal{L}\{f''\} = s^2F(s) - sf(0) - f'(0)$ |
| Convolution | $\mathcal{L}\{f*g\} = F(s)G(s)$ |
| Frequency shift | $\mathcal{L}\{e^{at}f\} = F(s-a)$ |

**Transfer function:** For $ay'' + by' + cy = u(t)$ with zero ICs, $H(s) = \frac{Y(s)}{U(s)} = \frac{1}{as^2 + bs + c}$.

In [ ]:
def bode_plot(numerator_coeffs, denominator_coeffs, omega_range=(0.01, 100), ax=None):
    """
    Compute and plot Bode magnitude and phase for a transfer function H(s).
    Coefficients are given in descending powers of s.
    """
    omega = np.logspace(np.log10(omega_range[0]), np.log10(omega_range[1]), 500)
    s = 1j * omega
    H = np.polyval(numerator_coeffs, s) / np.polyval(denominator_coeffs, s)

    if ax is None:
        fig, ax = plt.subplots(2, 1, figsize=(9, 6))

    ax[0].semilogx(omega, 20*np.log10(np.abs(H)), 'b-', lw=2)
    ax[0].set_ylabel('Magnitude (dB)'); ax[0].set_title('Bode Plot')
    ax[0].grid(True, alpha=0.3)

    ax[1].semilogx(omega, np.degrees(np.angle(H)), 'r-', lw=2)
    ax[1].set_xlabel('Frequency $\\omega$ (rad/s)')
    ax[1].set_ylabel('Phase (°)')
    ax[1].grid(True, alpha=0.3)
    return omega, H


# --- Damped oscillator transfer function: H(s) = 1 / (s^2 + 2*gamma*s + omega0^2) ---
fig, axes = plt.subplots(2, 1, figsize=(9, 7))
gamma_values = [0.1, 0.5, 1.0, 2.0]
omega0 = 5.0
colors = cm.plasma(np.linspace(0.15, 0.85, len(gamma_values)))

for gamma, c in zip(gamma_values, colors):
    num = [1.0]
    den = [1.0, 2*gamma, omega0**2]
    omega = np.logspace(-1, 2, 500)
    s = 1j * omega
    H = np.polyval(num, s) / np.polyval(den, s)

    axes[0].semilogx(omega, 20*np.log10(np.abs(H)), color=c, lw=2, label=f'$\\gamma={gamma}$')
    axes[1].semilogx(omega, np.degrees(np.angle(H)), color=c, lw=2, label=f'$\\gamma={gamma}$')

for ax in axes:
    ax.axvline(omega0, color='k', ls='--', lw=1, alpha=0.5)
    ax.legend(frameon=False, fontsize=8); ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Magnitude (dB)'); axes[0].set_title(f'Bode Plot — $\\omega_0={omega0}$')
axes[1].set_xlabel('$\\omega$ (rad/s)'); axes[1].set_ylabel('Phase (°)')
plt.tight_layout(); plt.show()

---

## 5. Exercises

1. **(Fourier series)** Derive the Fourier series of the sawtooth wave $f(x) = x$ on $[-\pi, \pi]$ analytically. Plot partial sums for $N = 1, 5, 15, 50$ and measure the Gibbs overshoot.

2. **(Parseval's theorem)** Verify numerically that $\sum_{n=-\infty}^{\infty}|X_n|^2 = \frac{1}{N}\sum_{k}|x_k|^2$ for a random signal.

3. **(FFT from scratch)** Implement an **iterative** (non-recursive) Cooley-Tukey FFT using bit-reversal permutation. Compare speed with the recursive version.

4. **(Convolution theorem)** Use the FFT to multiply two degree-100 polynomials. Compare the running time to the naive $O(N^2)$ polynomial multiplication.

5. **(Laplace)** Solve $y'' + 4y = \sin(2t)$, $y(0) = y'(0) = 0$ (resonance) using the Laplace transform. Verify the result numerically and explain the linear growth in amplitude.